### Links and Resources
| Concept | Reference |
|---|---|
| [Catalyst optimizer](https://databricks.com/glossary/catalyst-optimizer) | What `EXPLAIN` shows you |
| [Photon engine](https://www.databricks.com/product/photon) | When/why built-ins go vectorized C++ |
| [`noop` sink](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.format.html) | Forces execution without writing data |
| [Spark UI](https://spark.apache.org/docs/latest/web-ui.html) | The other side of perf debugging |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Performance: Built-in vs Python UDF vs Pandas UDF

The folklore is "built-ins are fastest, Pandas UDFs are middle, Python UDFs are worst." This notebook **measures it on your cluster** and shows you how to read the query plan to understand why.

We'll:

1. Run the same logic four ways: built-in, Python UDF, Arrow UDF, Pandas UDF.
2. Time each with the `noop` sink (no I/O — pure compute).
3. Read `EXPLAIN` to see what Catalyst does with each.
4. Talk about Photon and what disables it.

## Setup — a workload big enough to matter

In [ ]:
import time
import numpy as np, pandas as pd

np.random.seed(0)
N = 2_000_000

pdf = pd.DataFrame({
    "id":    np.arange(N),
    "value": np.random.randn(N) * 10 + 50,
    "name":  np.random.choice(["alice", "BOB", "Charlie", " Dana "], size=N),
})
df = spark.createDataFrame(pdf).repartition(8).cache()
df.count()              # materialize the cache
print("rows:", N)

## A reusable timer

The `noop` sink is the canonical way to benchmark Spark: it runs the full plan but writes nothing. We discard a warm-up run because the first call may include compilation / file-listing / cache priming.

In [ ]:
def bench(label, df_expr, runs=3):
    df_expr.write.format("noop").mode("overwrite").save()  # warm-up
    times = []
    for _ in range(runs):
        t = time.time()
        df_expr.write.format("noop").mode("overwrite").save()
        times.append(time.time() - t)
    print(f"  {label:24s} median={np.median(times):5.2f}s  min={min(times):5.2f}s  (n={runs})")

## Workload 1 — numeric: standardize a column

`(value - 50) / 10` four ways.

In [ ]:
from pyspark.sql.functions import udf, pandas_udf

@udf("double")
def py_norm(v):
    return None if v is None else (v - 50) / 10

@udf("double", useArrow=True)
def arrow_norm(v):
    return None if v is None else (v - 50) / 10

@pandas_udf("double")
def pd_norm(s: pd.Series) -> pd.Series:
    return (s - 50) / 10

print("=== numeric standardize ===")
bench("built-in",   df.select((F.col("value") - 50) / 10))
bench("Python UDF", df.select(py_norm("value")))
bench("Arrow UDF",  df.select(arrow_norm("value")))
bench("Pandas UDF", df.select(pd_norm("value")))

**Expected ordering** (your numbers will differ):

```
built-in   <  Pandas UDF  <  Arrow UDF  <  Python UDF
```

Built-ins win because Catalyst lowers them into Photon C++ kernels that operate on entire columns. Python UDFs lose because every row pays a serialization round-trip.

## Workload 2 — string: trim + title-case

`F.initcap(F.trim(...))` four ways.

In [ ]:
@udf("string")
def py_clean(s):
    return None if s is None else s.strip().title()

@udf("string", useArrow=True)
def arrow_clean(s):
    return None if s is None else s.strip().title()

@pandas_udf("string")
def pd_clean(s: pd.Series) -> pd.Series:
    return s.str.strip().str.title()

print("=== string clean ===")
bench("built-in",   df.select(F.initcap(F.trim("name"))))
bench("Python UDF", df.select(py_clean("name")))
bench("Arrow UDF",  df.select(arrow_clean("name")))
bench("Pandas UDF", df.select(pd_clean("name")))

## Reading the query plan — `EXPLAIN`

The plan tells you what Catalyst will actually run. Three levels of detail:

- `df.explain()` — physical plan only.
- `df.explain(True)` — parsed → analyzed → optimized → physical.
- `df.explain("formatted")` — node-by-node, easier to read.

In [ ]:
print("--- built-in ---")
df.select(F.initcap(F.trim("name"))).explain()

print("--- Python UDF ---")
df.select(py_clean("name")).explain()

**What to look for:**

- **`Project [initcap(trim(name#X))]`** in the built-in plan — it's a normal projection. Catalyst can fuse it with neighbours.
- **`BatchEvalPython`** in the Python UDF plan — that's the Python worker pipe. Photon can't go through it; the data exits to Python and comes back. Anything downstream of `BatchEvalPython` runs in standard Spark, not Photon, even if it could otherwise.

This is why a single Python UDF can de-optimize an *entire* query — not just its own column.

## Photon — when it kicks in (and when it doesn't)

Photon is a vectorized C++ engine that replaces Catalyst's whole-stage codegen for supported operations. To use it:

- Cluster must be **Photon-enabled** (a tickbox at cluster creation, or any serverless SQL warehouse).
- The plan must contain only **Photon-supported operators**. A Python/Pandas UDF or unsupported expression forces a fall-back to non-Photon for that stage.

Check `EXPLAIN` for nodes prefixed with `Photon` (e.g. `PhotonProject`, `PhotonScan`). If you see `Project` instead, Photon was disabled for that node.

In [ ]:
# This will show Photon nodes only on a Photon-enabled cluster (incl. serverless).
df.select(F.initcap(F.trim("name"))).explain("formatted")

## What slows things down besides UDFs

A UDF isn't the only way to lose performance. Common culprits:

- **`collect()` / `toPandas()`** on a big DataFrame — pulls the whole thing to the driver.
- **`count()` in a loop** — recomputes the plan each time.
- **`distinct()` then `count()`** — use `approx_count_distinct` if approximate is OK.
- **`groupBy().count()` with high-cardinality key** — fine, but if you only need top-N, use a window with `row_number`.
- **Large `collect_list`** — moves entire group to one executor.
- **Skewed joins** on a hot key — salt the key or enable AQE skew handling.

## Cleanup

In [ ]:
df.unpersist()
print("unpersisted")

## Takeaways

- **Built-in > Pandas UDF > Arrow UDF > Python UDF** — and the gaps are often 5–50×.
- A single Python UDF can disable Photon for the rest of the stage. That's a bigger cost than just the UDF itself.
- Always benchmark with the `noop` sink and a warm-up run. `count()` works too but introduces extra shuffle.
- `EXPLAIN` shows `BatchEvalPython` for any node that round-trips to Python — make that your "is this fast?" smoke test.
- Tune *before* you write the Python: most "needs a UDF" cases turn out to be expressible in built-ins.